# Зонд окружения JupyterHub

Запусти все ячейки сверху вниз (`Kernel → Restart & Run All`) и пришли вывод целиком.

Ничего не устанавливает и не меняет — только читает. Использует лишь стандартную
библиотеку Python, поэтому работает на любом ядре.

In [ ]:
import os, sys, platform, shutil, subprocess, socket, textwrap

def run(cmd, timeout=30):
    """Выполнить команду, вернуть (код возврата, вывод). Не падает."""
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return p.returncode, (p.stdout + p.stderr).strip()
    except FileNotFoundError:
        return 127, "команда не найдена"
    except subprocess.TimeoutExpired:
        return 124, f"таймаут {timeout}s"
    except Exception as e:
        return 1, f"{type(e).__name__}: {e}"

def section(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

print("зонд готов")

## 1. Кто и где я

In [ ]:
section("1. СИСТЕМА")
print("hostname   :", socket.gethostname())
print("платформа  :", platform.platform())
print("python     :", sys.version.replace("\n", " "))
print("исполняемый:", sys.executable)
print("пользователь:", os.environ.get("USER") or os.environ.get("LOGNAME") or "?")
print("домашняя   :", os.path.expanduser("~"))
print("рабочая    :", os.getcwd())

rc, out = run("cat /etc/os-release 2>/dev/null | head -3")
print("\nдистрибутив:\n" + (out or "неизвестно"))

# Признаки того, что мы внутри JupyterHub / контейнера
jh = {k: v for k, v in os.environ.items() if k.startswith(("JUPYTER", "JPY", "NB_"))}
print("\nпеременные JupyterHub:", jh if jh else "нет")
print("внутри контейнера:", os.path.exists("/.dockerenv"))

## 2. Видеокарта и драйвер

Самое важное: модель карты, объём памяти и версия драйвера. От версии драйвера зависит, какую сборку CUDA можно поставить.

In [ ]:
section("2. GPU")
rc, out = run("nvidia-smi")
if rc == 0:
    print(out)
else:
    print("nvidia-smi недоступен (код %s):" % rc)
    print(out)
    print("\n>>> без GPU обучение невозможно; уточни у администратора лаборатории")

rc, out = run("nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv")
if rc == 0:
    print("\nкратко:\n" + out)

## 3. Диск

Нужно порядка 100–200 ГБ: окружение с CUDA-библиотеками, ассеты ManiSkill, чекпоинты.

In [ ]:
section("3. ДИСК")
home = os.path.expanduser("~")
for path in [home, os.getcwd(), "/tmp", "/workspace", "/scratch", "/data"]:
    if os.path.isdir(path):
        try:
            u = shutil.disk_usage(path)
            print(f"{path:<14} всего {u.total/2**30:7.1f} ГБ | свободно {u.free/2**30:7.1f} ГБ")
        except Exception as e:
            print(f"{path:<14} ошибка: {e}")

rc, out = run("quota -s 2>/dev/null || echo 'квоты не настроены или команда недоступна'")
print("\nквота:\n" + out)

print("\nчто уже лежит в домашней:")
rc, out = run(f"ls -la {home} | head -20")
print(out)

## 4. Инструменты

От наличия `git`, компилятора и менеджера пакетов зависит способ развёртывания.

In [ ]:
section("4. ИНСТРУМЕНТЫ")
tools = ["git", "curl", "wget", "uv", "pip", "pip3", "conda", "mamba",
         "gcc", "g++", "make", "tar", "unzip", "nohup", "setsid", "tmux", "screen", "bash"]
for t in tools:
    p = shutil.which(t)
    print(f"  {t:<8} {'✓ ' + p if p else '✗ нет'}")

rc, out = run("python -m venv --help > /dev/null 2>&1 && echo ok || echo нет")
print("\nmodule venv:", out)

rc, out = run("pip --version")
print("pip        :", out if rc == 0 else "недоступен")

## 5. Интернет

Если исходящих соединений нет, зависимости придётся привозить архивом — это меняет весь план развёртывания.

In [ ]:
section("5. СЕТЬ")
hosts = [("pypi.org", 443), ("files.pythonhosted.org", 443),
         ("github.com", 443), ("download.pytorch.org", 443),
         ("huggingface.co", 443)]
for host, port in hosts:
    try:
        socket.create_connection((host, port), timeout=6).close()
        print(f"  {host:<28} ✓ доступен")
    except Exception as e:
        print(f"  {host:<28} ✗ {type(e).__name__}")

print("\nпрокси в окружении:")
proxies = {k: v for k, v in os.environ.items() if "proxy" in k.lower()}
print("  ", proxies if proxies else "не заданы")

rc, out = run("pip config list 2>/dev/null || true")
print("\nнастройки pip (возможен внутренний индекс):\n  " + (out or "пусто"))

## 6. Что уже установлено

Если в образе уже есть подходящий PyTorch, это экономит несколько гигабайт загрузки.

In [ ]:
section("6. ПАКЕТЫ")
for mod in ["torch", "gymnasium", "mani_skill", "sapien", "torchrl", "benchmarl",
            "tensorboard", "numpy"]:
    try:
        m = __import__(mod)
        print(f"  {mod:<12} ✓ {getattr(m, '__version__', 'версия неизвестна')}")
    except Exception as e:
        print(f"  {mod:<12} ✗ {type(e).__name__}")

try:
    import torch
    print("\n  torch.version.cuda      :", torch.version.cuda)
    print("  torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("  устройство              :", torch.cuda.get_device_name(0))
        print("  compute capability      :", torch.cuda.get_device_capability(0))
        # is_available() бывает True на сломанной сборке — проверяем реальным умножением
        x = torch.randn(512, 512, device="cuda")
        print("  реальный matmul на GPU  : ок,", float((x @ x).sum()))
except Exception as e:
    print("\n  проверка CUDA не прошла:", type(e).__name__, e)

## 7. Живучесть процессов

Обучение идёт часами. Ядро ноутбука умирает при закрытии вкладки, поэтому важно, можно ли запустить фоновый процесс, переживающий отключение.

In [ ]:
section("7. ФОНОВЫЕ ПРОЦЕССЫ")
rc, out = run("cd /tmp && nohup sleep 5 > /tmp/_probe_nohup.log 2>&1 & echo запущен $!")
print("nohup:", out)

rc, out = run("ps -eo pid,etime,cmd --no-headers | head -15")
print("\nпроцессы пользователя:\n" + out)

# Есть ли в JupyterHub терминал — если да, всё делается обычными командами
try:
    import jupyter_server_terminals  # noqa: F401
    print("\nтерминал JupyterHub: ✓ модуль установлен (вкладка Terminal должна быть)")
except Exception:
    print("\nтерминал JupyterHub: ✗ модуль jupyter_server_terminals не найден")
    print("   (проверь вручную: File → New → Terminal)")

rc, out = run("ulimit -a")
print("\nограничения оболочки:\n" + out)

## 8. Итог

In [ ]:
section("ИТОГ — пришли этот блок целиком")
rc, gpu = run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
free = shutil.disk_usage(os.path.expanduser("~")).free / 2**30
net = []
for host in ["pypi.org", "github.com"]:
    try:
        socket.create_connection((host, 443), timeout=6).close()
        net.append(host + ":да")
    except Exception:
        net.append(host + ":нет")

print("GPU        :", gpu if rc == 0 else "НЕТ")
print("Свободно   : %.1f ГБ в домашней" % free)
print("Сеть       :", ", ".join(net))
print("git        :", "есть" if shutil.which("git") else "НЕТ")
print("uv         :", "есть" if shutil.which("uv") else "нет (поставится при развёртывании)")
print("python     :", sys.version.split()[0])
try:
    import torch
    print("torch      :", torch.__version__, "| CUDA", torch.version.cuda,
          "| GPU доступен:", torch.cuda.is_available())
except Exception:
    print("torch      : не установлен")